## 📝 Spark Architecture — key terms

Before talking about partitions and joins, know who does what in a Spark cluster:

| Term | What it is |
|------|-----------|
| **Driver** | The process running *your* code (this notebook). It creates the `SparkSession`, builds the query plan, and coordinates the work — but does not process the data itself. |
| **Master** (cluster manager) | The resource negotiator. The driver asks it: *"I need CPU + memory"* — the master finds space on the workers and launches executors there. (In standalone Spark it's the Master process; on YARN/Kubernetes/Databricks the cluster manager plays this role.) |
| **Worker** | A machine (node) in the cluster that offers its CPU cores and memory. Workers host executors. |
| **Executor** | A JVM process launched on a worker that does the actual data processing. It runs **tasks** (one per CPU core it owns) and caches data in memory. |
| **Task** | The smallest unit of work — **one task processes one partition** of the data on one executor core. |

```text
       ┌──────────┐
       │  Driver  │  ← this notebook (SparkSession)
       └────┬─────┘
            │ requests resources
       ┌────▼─────┐
       │  Master  │  ← cluster manager
       └────┬─────┘
     ┌──────┴────────┐
┌────▼─────┐    ┌────▼─────┐
│ Worker 1 │    │ Worker 2 │        ← machines
│┌────────┐│    │┌────────┐│
││Executor││    ││Executor││        ← JVM processes
││ task───┼┼─┐  ││ task   ││        ← 1 task = 1 partition = 1 core
│└────────┘│ │  │└────────┘│
└──────────┘ │  └──────────┘
             └── each task chews through one partition
```

### What does the partition number mean?

A DataFrame is split into **partitions** — independent chunks of rows. The partition count (`df.rdd.getNumPartitions()`) is the **maximum parallelism**: 20 partitions means the work can be split into 20 tasks running at the same time (if enough cores exist). With 2 cores and 20 partitions, tasks run 2 at a time in 10 waves.

### What is shuffling?

A **shuffle** is Spark physically **moving rows between partitions/executors over the network**, so that rows that belong together end up together. It happens whenever an operation needs data grouped differently than it currently is:

- `groupBy()` / `agg()` — all rows of a key must meet in one place to be aggregated
- `join()` — matching keys from both DataFrames must land in the same partition
- `orderBy()`, `distinct()`, `repartition()`

Shuffles are the **most expensive** thing in Spark (network + disk I/O — executors write shuffle files that others fetch), which is why partitioning strategy matters. After a shuffle, Spark SQL uses `spark.sql.shuffle.partitions` partitions (default **200**).

### Where does *this* course run?

- `master("local[*]")` (these notebooks) — no real cluster: driver + executor live in **one JVM**, `*` = use all CPU cores of your machine.
- [docker-images/](docker-images/) setup — a real standalone cluster: **1 master + 2 workers**.
- Databricks Community Edition (chapter 6 tip) — a single-node cluster: the driver does everything with its 2 cores.

> 📎 **Deep dive:** the table above is the short version. For the full story — what *submitting* a program means (`spark-submit`), what a JVM is, the 6-step driver ↔ cluster-manager ↔ executor lifecycle, how executors/cores/tasks add up, what "success or failure" means, the 4 cluster managers, and **client vs cluster deployment mode** — see the separate note [spark-architecture-notes.ipynb](spark-architecture-notes.ipynb).

In [49]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("Joins and Data Partitions").master("local[*]").getOrCreate())

In [50]:
# Emp Data & Schema
emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","Female","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]
emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

dept_data = [
    ["101", "Sales", "NYC", "US", "1000000"],
    ["102", "Marketing", "LA", "US", "900000"],
    ["103", "Finance", "London", "UK", "1200000"],
    ["104", "Engineering", "Beijing", "China", "1500000"],
    ["105", "Human Resources", "Tokyo", "Japan", "800000"],
    ["106", "Research and Development", "Perth", "Australia", "1100000"],
    ["107", "Customer Service", "Sydney", "Australia", "950000"]
]
dept_schema = "department_id string, department_name string, city string, country string, budget string"

In [51]:
# Create emp & dept DataFrame
emp = spark.createDataFrame(data=emp_data, schema=emp_schema)
dept = spark.createDataFrame(data=dept_data, schema=dept_schema)

# Print Schema
emp.printSchema()
dept.printSchema()

# Show emp dataframe (ACTION)
emp.show(truncate=False)
dept.show(truncate=False)

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)

root
 |-- department_id: string (nullable = true)
 |-- department_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- budget: string (nullable = true)

+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|name         |age|gender|salary|hire_date |
+-----------+-------------+-------------+---+------+------+----------+
|001        |101          |John Doe     |30 |Male  |50000 |2015-01-01|
|002        |101          |Jane Smith   |25 |Female|45000 |2016-02-15|
|003        |102          |Bob Brown    |35 |Male  |55000 |2014-05-01|
|004        |102          |Alice Lee    |28 |Female|48000 |2017-09-30|
|005      

#--------------------------------------------------------------------------------------------------------
## 📝 What is repartitioning?

Repartitioning = changing **how many** partitions the data is split into, and/or **which rows go to which partition**.

| Method | What it does | Cost |
|--------|-------------|------|
| `repartition(n)` | Reshuffle into exactly `n` partitions | Full shuffle — expensive, but evenly balanced |
| `repartition(n, col)` | Shuffle so rows with the same value of `col` land in the same partition (e.g. `department_id`) | Full shuffle — but pre-positions data for a later `join()`/`groupBy()` on that column |
| `coalesce(n)` | Only **reduce** the partition count by merging neighbours (cannot increase only decrease is possible) | No full shuffle — cheap, but partitions can end up uneven |

**Why it matters:**

- too **few** partitions → cores sit idle, no parallelism
- too **many** partitions → scheduling overhead, many tiny files on write
- joins/groupBy shuffle data by key anyway — pre-partitioning by that key can avoid extra shuffles

In [52]:
# check current partition count
print("emp partitions :", emp.rdd.getNumPartitions())
print("dept partitions:", dept.rdd.getNumPartitions())

# full shuffle into 4 partitions
emp_repart = emp.repartition(4)
print("after repartition(4)      :", emp_repart.rdd.getNumPartitions())

# shuffle by column — all rows of a department end up in the same partition
emp_by_dept = emp.repartition(4, "department_id")
print("after repartition(4, col) :", emp_by_dept.rdd.getNumPartitions())

# shrink without a full shuffle
emp_small = emp_repart.coalesce(2)
print("after coalesce(2)         :", emp_small.rdd.getNumPartitions())

emp partitions : 20
dept partitions: 20
after repartition(4)      : 4
after repartition(4, col) : 4
after coalesce(2)         : 2


In [53]:
# spark_partition_id() returns the partition ID of each row —
# useful to see how rows are distributed across partitions
# (e.g. verify repartition(4, "department_id") put each department in one partition).

from pyspark.sql.functions import spark_partition_id

emp1 = emp.withColumn("partition_id", spark_partition_id())
print(f"Number of partitions: {emp1.rdd.getNumPartitions()}")
emp1.show(truncate=False)

Number of partitions: 20
+-----------+-------------+-------------+---+------+------+----------+------------+
|employee_id|department_id|name         |age|gender|salary|hire_date |partition_id|
+-----------+-------------+-------------+---+------+------+----------+------------+
|001        |101          |John Doe     |30 |Male  |50000 |2015-01-01|0           |
|002        |101          |Jane Smith   |25 |Female|45000 |2016-02-15|1           |
|003        |102          |Bob Brown    |35 |Male  |55000 |2014-05-01|2           |
|004        |102          |Alice Lee    |28 |Female|48000 |2017-09-30|3           |
|005        |103          |Jack Chan    |40 |Male  |60000 |2013-04-01|4           |
|006        |103          |Jill Wong    |32 |Female|52000 |2018-07-01|5           |
|007        |101          |James Johnson|42 |Male  |70000 |2012-03-15|6           |
|008        |102          |Kate Kim     |29 |Female|51000 |2019-10-01|7           |
|009        |103          |Tom Tan      |33 |Male  

In [54]:
# repartitioning the same by department_id and 4 partitions
from pyspark.sql.functions import spark_partition_id

emp1 = emp.repartition(4, "department_id").withColumn("partition_id", spark_partition_id())
print(f"Number of partitions: {emp1.rdd.getNumPartitions()}")
emp1.show(truncate=False)

# here all 102 department rows should be in the same partition, and all 103 department rows should be in the same partition, etc.

Number of partitions: 4
+-----------+-------------+-------------+---+------+------+----------+------------+
|employee_id|department_id|name         |age|gender|salary|hire_date |partition_id|
+-----------+-------------+-------------+---+------+------+----------+------------+
|003        |102          |Bob Brown    |35 |Male  |55000 |2014-05-01|0           |
|004        |102          |Alice Lee    |28 |Female|48000 |2017-09-30|0           |
|008        |102          |Kate Kim     |29 |Female|51000 |2019-10-01|0           |
|014        |107          |Emily Lee    |26 |Female|46000 |2019-01-01|0           |
|016        |107          |Kelly Zhang  |30 |Female|49000 |2018-04-01|0           |
|020        |102          |Grace Kim    |32 |Female|53000 |2018-11-01|0           |
|012        |105          |Susan Chen   |31 |Female|54000 |2017-02-15|1           |
|017        |105          |George Wang  |34 |Male  |57000 |2016-03-15|1           |
|010        |104          |Lisa Lee     |27 |Female|

In [55]:
# INNER JOIN datasets
'''SELECT e.name, d.department_name, d.department_id, e.salary
FROM emp e INNER JOIN dept d ON e.department_id = d.department_id'''

# In the SQL above, "e" and "d" are table aliases declared by "FROM emp e ... dept d".
# In the PySpark code below we instead qualify columns with the DataFrame variables
# (emp.name, dept.department_name). Qualifying matters because BOTH DataFrames have a
# department_id column — an unqualified reference would be ambiguous after the join.
# (PySpark's equivalent of SQL aliases: emp.alias("e") lets you write col("e.name") —
# handy when chains of joins make full DataFrame names long.)

df_joined = emp.join(dept, how="inner", on=emp.department_id == dept.department_id)
df_joined.select(emp.name, dept.department_name, dept.department_id, emp.salary).show()

+-------------+--------------------+-------------+------+
|         name|     department_name|department_id|salary|
+-------------+--------------------+-------------+------+
|     John Doe|               Sales|          101| 50000|
|   Jane Smith|               Sales|          101| 45000|
|James Johnson|               Sales|          101| 70000|
|    Bob Brown|           Marketing|          102| 55000|
|    Alice Lee|           Marketing|          102| 48000|
|     Kate Kim|           Marketing|          102| 51000|
|    Grace Kim|           Marketing|          102| 53000|
|    Jack Chan|             Finance|          103| 60000|
|    Jill Wong|             Finance|          103| 52000|
|      Tom Tan|             Finance|          103| 58000|
|  Steven Chen|             Finance|          103| 62000|
|     Lisa Lee|         Engineering|          104| 47000|
|   David Park|         Engineering|          104| 65000|
|    Nancy Liu|         Engineering|          104| 50000|
|   Susan Chen

In [61]:
# Bonus TIP — Joins with cascading conditions
# the on= condition doesn't have to be a single equality: chain any Column checks
# with & (AND) / | (OR), each condition wrapped in its own parentheses.

# Join with department_id and only for departments 101 & 102
'''SELECT * FROM emp e INNER JOIN dept d
ON e.department_id = d.department_id AND e.department_id IN ('101', '102') AND e.department_id IS NOT NULL'''

df_cascade = emp.join(
    dept,
    how="inner",
    on=(emp.department_id == dept.department_id) & ((emp.department_id =="101") | (emp.department_id == "102"))
    & (emp.department_id.isNotNull())
)
df_cascade.show()

# Join with not null / null conditions
'''SELECT * FROM emp e INNER JOIN dept d
ON e.department_id = d.department_id AND e.department_id IS  NULL'''

df_notnull = emp.join(
    dept,
    how="inner",
    on=(emp.department_id == dept.department_id) & (emp.department_id.isNull())
)
df_notnull.show()

+-----------+-------------+-------------+---+------+------+----------+-------------+---------------+----+-------+-------+
|employee_id|department_id|         name|age|gender|salary| hire_date|department_id|department_name|city|country| budget|
+-----------+-------------+-------------+---+------+------+----------+-------------+---------------+----+-------+-------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|          101|          Sales| NYC|     US|1000000|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|          101|          Sales| NYC|     US|1000000|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|          101|          Sales| NYC|     US|1000000|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|          102|      Marketing|  LA|     US| 900000|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|          102|      Marketing|  LA|     US| 900000|
|        008|          1